In [ ]:
import numpy as np
import pandas as pd


import matplotlib.pyplot as plt
from PIL import Image
import os

import tensorflow as tf
from tensorflow.keras.callbacks import TensorBoard
import datetime

print(os.getcwd())

In [ ]:
import logging
import smtplib

use_mail = False

logger = logging.getLogger(__name__)
logger.setLevel(logging.INFO)

formatter = logging.Formatter('%(name)s : %(levelname)s:%(levelno)s  %(asctime)s    %(message)s ')

info_handler = logging.FileHandler('Logs/Advanced_Logging_INFO_Level.log')
info_handler.setLevel(logging.INFO)
info_handler.setFormatter(formatter)
logger.addHandler(info_handler)



if use_mail:
    from email_logger_config import email_info
    login = "your@gmail.com"
    pswd = "**** **** **** ****" # Configured over gmail app-password
    email_info_example = {
    "mailhost": ('smtp.gmail.com', 587), # Everywhere on teh internet it recommended port 465, but what ended working was port 587
    "fromaddr": ["your@gmail.com"],
    "toaddrs": ["your@gmail.com"],
    "subject": 'Script Error Message',
    "credentials": (login, pswd),
    "secure": (),
    "timeout": 15.0 # time-out  default is 1 second, which leads to a time-out error
    }

    error_emailer = logging.handlers.SMTPHandler(**email_info)
    
    error_emailer.setLevel(logging.ERROR)
    logger.addHandler(error_emailer)
else:
    error_handler = logging.FileHandler('Logs/Advanced_Logging_ERROR_Level.log')
    error_handler.setLevel(logging.ERROR)
    error_handler.setFormatter(formatter)
    logger.addHandler(error_handler)

# Hypothetical


logger.info("Loggers created")

In [ ]:
google_colab = tf.config.list_physical_devices('GPU') # empty list equals Fasle
if google_colab:
    # allows acces to the google drive hwere the data is saved
    from google.colab import drive
    drive.mount('/content/drive')
    root_dir = "./drive/MyDrive/Colab Data/ResidualNeuralNetwork"
    print("Num GPUs Available: ", len(tf.config.list_physical_devices('GPU')))

    import timeit

    device_name = tf.test.gpu_device_name()
    if device_name != '/device:GPU:0':
      print(
          '\n\nThis error most likely means that this notebook is not '
          'configured to use a GPU.  Change this in Notebook Settings via the '
          'command palette (cmd/ctrl-shift-P) or the Edit menu.\n\n')
      raise SystemError('GPU device not found')
    
    def cpu():
      with tf.device('/cpu:0'):
        random_image_cpu = tf.random.normal((100, 100, 100, 3))
        net_cpu = tf.keras.layers.Conv2D(32, 7)(random_image_cpu)
        return tf.math.reduce_sum(net_cpu)
    
    def gpu():
      with tf.device('/device:GPU:0'):
        random_image_gpu = tf.random.normal((100, 100, 100, 3))
        net_gpu = tf.keras.layers.Conv2D(32, 7)(random_image_gpu)
        return tf.math.reduce_sum(net_gpu)
    
    # We run each op once to warm up; see: https://stackoverflow.com/a/45067900
    cpu()
    gpu()
    
    # Run the op several times.
    print('Time (s) to convolve 32x7x7x3 filter over random 100x100x100x3 images '
          '(batch x height x width x channel). Sum of ten runs.')
    print('CPU (s):')
    cpu_time = timeit.timeit('cpu()', number=10, setup="from __main__ import cpu")
    print(cpu_time)
    print('GPU (s):')
    gpu_time = timeit.timeit('gpu()', number=10, setup="from __main__ import gpu")
    print(gpu_time)
    print('GPU speedup over CPU: {}x'.format(int(cpu_time/gpu_time)))

    logger.info("Using GPU")
else:
    root_dir = "."
    logger.info("Using CPU")

In [ ]:
# prepare the Data and DataAugmentation
batch_size = 50
height = 256
width = 256
classes = 2

ds_train = tf.keras.preprocessing.image_dataset_from_directory(
    f"{root_dir}/Data/Cats_Dogs/train", # The Difectory that contains the images. EAch Image class has its own subfolder
    labels = "inferred", # labels are generated from directory structure, i.e. each subfolder = 1 label
    label_mode = "categorical", #one-hot encoding
    class_names = ["Cat", "Dog"], # only when labels = 'inferred'. Determines the order
    color_mode = "rgb",
    batch_size = batch_size,
    image_size = (height, width), # Default. Images are Resized to that size.
    shuffle = True,
    seed = 42, # important so that training and validationsplit is euqal each time. There fore I can copy this code to receive teh validation set
    validation_split = 0.1,
    subset = "training" # specify that I want the 1-validaion_split training data
)

ds_val = tf.keras.preprocessing.image_dataset_from_directory(
    f"{root_dir}/Data/Cats_Dogs/train", # The Difectory that contains the images. EAch Image class has its own subfolder
    labels = "inferred", # labels are generated from directory structure, i.e. each subfolder = 1 label
    label_mode = "categorical", #one-hot encoding
    class_names = ["Cat", "Dog"], # only when labels = 'inferred'. Determines the order
    color_mode = "rgb",
    batch_size = batch_size,
    image_size = (height, width), # Default. Images are Resized to that size.
    shuffle = True,
    seed = 42, # important so that training and validationsplit is euqal each time. There fore I can copy this code to receive teh validation set
    validation_split = 0.1,
    subset = "validation" # specify that I want the 1-validaion_split training data
)

In [ ]:
earlyStop = tf.keras.callbacks.EarlyStopping(monitor='val_loss', patience=3, min_delta = 1e-10, restore_best_weights=True)

# checkpPointModel: Saves the BEST model for when the training is finished to simply reload the model
checkPointModel = tf.keras.callbacks.ModelCheckpoint(f"{root_dir}/Checkpoints" + "/RNN_Dashboard_Test_BestModel.keras", monitor='val_loss', verbose=0, save_best_only=True, save_weights_only=False, mode='auto', save_freq='epoch', initial_value_threshold=None)
# checkpPointTraining: Saves the weights per epoch to continue training in case of interruptions.
checkPointTraining = tf.keras.callbacks.ModelCheckpoint(f"{root_dir}/Checkpoints" + "/RNN_Dashboard_Test_{epoch}_test.keras", monitor='val_loss', verbose=0, save_best_only=False, save_weights_only=False, mode='auto', save_freq='epoch', initial_value_threshold=None)





# TensorBoard loagging and real-time survaillance
log_dir = f"{root_dir}/Checkpoints/TensorBoard_Checkpoint"

tensorboard_callback = TensorBoard(
    log_dir=log_dir,
    histogram_freq=1,  # Log weight histograms every epoch
    update_freq='epoch',  # Update after each epoch
    profile_batch=0  # Disable profiling for faster training
)

In [ ]:
# Start TensorBoard server once trainign has started over the anaconda prompter
###### tensorboard --logdir=Checkpoints

# Open browser to http://localhost:6006
# View real-time loss curves, histograms, and model graphs

For simplicity and speeds sake, I left out further configurations like gradient clipping and label smothing

In [ ]:
# Build and Compile the Model
# strategy = tf.distribute.MultiWorkerMirroredStrategy()
# with strategy.scope():
kernel_regularizer=tf.keras.regularizers.l1_l2(l1=0, l2=1e-4)

Image_augmentation_pipeline = tf.keras.Sequential([
tf.keras.layers.Input(shape = (height, width, 3)),
tf.keras.layers.RandomBrightness(0.2),
tf.keras.layers.RandomFlip(mode = "horizontal_and_vertical"),
# tf.keras.layers.RandomCrop(height = list(range(200)), width = list(range(200))),
tf.keras.layers.Resizing(height = height,  width = width)
])

def ResidualBlock(inputs, filters, kernel_size):
    """
    Beim Add müssen der Input Tensor und X die gleiche Shape haben.
    2 Lösungen:
    1.) Das Letzte Conv2D befor Add hat so viele Channels/ Filters wie der Input Array
    2.) Mit einem 1x1 kernel das InputLayer auf die gew+nschte Layerzahl hochskalieren
    """

    # inputs = tf.keras.layers.Conv2D(filters = filters, kernel_size = (1, 1), padding = "same")(inputs)
    X = tf.keras.layers.Conv2D(filters = filters, kernel_size = kernel_size, padding = "same", input_shape = (height, width, 3), kernel_regularizer = kernel_regularizer)(inputs)
    X = tf.keras.layers.Dropout(0.2)(X)
    X = tf.keras.layers.BatchNormalization(axis = 3)(X)
    # X = tf.keras.layers.Add()([X, inputs])
    X = tf.keras.layers.Activation('relu')(X)
    X = tf.keras.layers.Conv2D(filters = 3, kernel_size = kernel_size, padding = "same", kernel_regularizer = kernel_regularizer)(X)
    X = tf.keras.layers.Dropout(0.2)(X)
    X = tf.keras.layers.BatchNormalization(axis = 3)(X)
    ##### Mein Fehler war hier. Die beiden Tensoren sollen kombiniert werden. Aus dem Grund kam es zu einem Fehler, da ser Input nur 3 channel hat und das X eine andere Anzahl an channeln hatte.
    X = tf.keras.layers.Add()([X, inputs])
    X = tf.keras.layers.Activation('relu')(X)
    return X


X_input = tf.keras.layers.Input(shape = (height, width, 3))
X = ResidualBlock(X_input, kernel_size = (3, 3), filters = 25)
# X = ResidualBlock(X, kernel_size = (3, 3), filters = 25)
X = tf.keras.layers.AveragePooling2D(pool_size = 3, padding = "same")(X)
X = tf.keras.layers.Flatten()(X)
X = tf.keras.layers.Dense(300, activation='relu', kernel_regularizer = kernel_regularizer)(X)
X = tf.keras.layers.Dense(classes, activation='sigmoid')(X)


ResNet = tf.keras.Model(inputs = X_input, outputs = X)
Training_Model = tf.keras.Sequential([Image_augmentation_pipeline, ResNet])


loss = tf.keras.losses.BinaryCrossentropy()
optimizer=tf.keras.optimizers.Adam(learning_rate=0.0005)
# Training_Model.compile(loss=loss, optimizer=optimizer, metrics=['accuracy'])

##########################################################################################################################

if not os.listdir(f"{root_dir}/Models/Tensorflow"):
    try:
        # Only do all of this when the Finished Model has not yet been generated
        epochs = 50
    
        SavePoints = [i for i in os.listdir(f"{root_dir}/Checkpoints") if i.startswith("RNN_Dashboard_Test") and i.endswith("_test.keras")]
        try:
            # Do checkpoints alerady exist ?
            initial_epoch = np.max([int(i.split("_")[-2]) for i in SavePoints])
        except:
            initial_epoch = 0
    
        print(initial_epoch)
    
        if initial_epoch == epochs:
            # laod best model, no training required
            TrainingModel = tf.keras.models.load_model(f'{root_dir}/Checkpoints/RNN_Dashboard_Test_BestModel.keras')
    
        else:
            # Either Start trainign from Scratch if no checkpoints exist or continue training from checkpoint
            if initial_epoch != 0:
                # For future cases: Only save just the weights if training is finsihed. Save the entire model when training is still ongoing. Or else previous progress is 'forgotten'
                Training_Model = tf.keras.models.load_model(f"{root_dir}/Checkpoints/RNN_Dashboard_Test_{initial_epoch}_test.keras")
            else:
                Training_Model.compile(loss=loss, optimizer=optimizer, metrics=['accuracy'])
            logger.info(f"Start Training at epoch {initial_epoch + 1}")
            history = Training_Model.fit(ds_train, epochs=epochs, verbose=1, validation_data=ds_val, callbacks=[earlyStop, checkPointModel, checkPointTraining, tensorboard_callback], initial_epoch = initial_epoch)
    except Exception as e:
        logger.error("Error during training")
        logger.warning(e)

    try:
        # Isolate the RESNet part of the model, drops the augmentation layers that was use dfor training.
        Training_Model = tf.keras.Model(inputs = Training_Model.layers[1].input, outputs = Training_Model.layers[1].output)
        # iamges need to be a specific size, thats why we still need a Resizing Layer for the finished network.
        Image_augmentation_pipeline = tf.keras.Sequential([
            tf.keras.layers.Resizing(height = height,  width = width)
        ])
        ResNet = tf.keras.Sequential([Image_augmentation_pipeline, Training_Model])
        ResNet.compile(loss=loss, optimizer=optimizer, metrics=['accuracy'])
        ResNet.build(input_shape = (None, height, width, 3))
        ResNet.summary()
        ResNet.save(f"{root_dir}/Models/Tensorflow/RNN_Dashboard_Test_BestModel.keras")
    except Exception as e:
        logging.error("Error during finishing the model")
        logging.warning(e)

In [ ]:
# Evaluate the finished model using Val Data
ResNet = tf.keras.models.load_model(f"{root_dir}/Models/Tensorflow/RNN_Dashboard_Test_BestModel.keras")
ResNet.evaluate(ds_val)

In [ ]:
# perform the prediction of the test data
ds_test = tf.keras.preprocessing.image_dataset_from_directory(
    directory = f"{root_dir}/Data/Cats_Dogs/test",
    labels = None,
    label_mode = None,
    color_mode = "rgb",
    batch_size = batch_size,
    image_size = (height, width), # Default. Images are Resized to that size.
    shuffle = True)

In [ ]:
Prediction = ResNet.predict(ds_test)

In [ ]:
print(Prediction)